# Creating clusters for the DeepNeuroBench Project

#### Please read carefully all the instructions. Specially the comments given in each cell. These comments are pertinent to run a cluster successfully. 

If the fabric environment is not set please read through the topic <code>FABRIC Environment Setup</code> in <code>start_here.ipynb</code> notebook given under <code>jupyter-example</code> folder in fabric's JupyterHub.

### STEP-1 Configuring FABRIC Credentials

In [ ]:
# Run this cell
import pandas as pd
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
try:
    fablib = fablib_manager()
    # fablib.show_config() #Uncomment to see FABlib Configurations
    resources = fablib.list_sites() #Uncomment to see available site/resources
except Exception as e:
    print(f"Exception: {e}")
pd.DataFrame(resources.data)[['Hosts','CPUs','Name','Cores Available','Ram Available','Tesla T4 Available','RTX6000 Available','A30 Available','A40 Available']] #Uncomment to see available resources in dataframe


### Step-2: Creating cluster (3-worker nodes + 1-master node)

#### Cluster (C1) - 64 cores, 64 GB RAM, 1000 GB Storage

In [ ]:
"""FABRIC cluster provisioning: create, configure, and provision a master/worker cluster.
Provisions a single-site cluster on the FABRIC testbed with one master node and three worker nodes connected via an L2 network. Handles IP assignment, inventory generation, 
/etc/hosts distribution, SSH key distribution, and IPv6 NAT64 setup.
Requirements:
    - FABRIC JupyterHub environment (or fabrictestbed-extensions installed with
      valid credentials at ~/.fabric/config).
    - nat64.sh present at WORK_DIR if any node has an IPv6 management address.
Usage (notebook cell):
    cluster = main()              # provision everything
    renew_slice("C1-HAWI", days=15)  # extend lease separately
"""
import subprocess
from datetime import datetime, timedelta, timezone
from ipaddress import IPv4Network, IPv6Address, ip_address
from pathlib import Path
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
# ===========================================================================
#  Configuration — edit these for each experiment
# ===========================================================================
NUM_NODES = 4
SLICE_NAME = "C1-HAWI"
SITE = "HAWI"
MASTER_TYPE = "fabric.c8.m16.d1000"
WORKER_TYPE = "fabric.c64.m64.d100"
IMAGE = "default_ubuntu_20"
NETWORK_NAME = "cluster_network"
SUBNET = IPv4Network("192.168.1.0/24")
GPU_VENDOR = "NVIDIA"

WORK_DIR = Path("/home/fabric/work")
HOSTS_FILE = WORK_DIR / "hosts"
IPS_FILE = WORK_DIR / "ips.txt"
WORKERS_FILE = WORK_DIR / "workers"
GPU_IPS_FILE = WORK_DIR / "gpu_ips.txt"
SSH_KEY = WORK_DIR / "id_rsa"
NAT64_SCRIPT = WORK_DIR / "nat64.sh"
# ===========================================================================
#  Initialise fablib
# ===========================================================================
fablib = fablib_manager()
# ===========================================================================
#  Helper Functions
# ===========================================================================
def _ordered_nodes(cluster):
    """Return nodes sorted by name so Node1 (master) is always index 0."""
    return sorted(cluster.get_nodes(), key=lambda n: n.get_name())
# ===========================================================================
#  Step 1 — Create the slice
# ===========================================================================
def create_slice():
    """Build a master + worker slice on a single FABRIC site and submit."""
    cluster = fablib.new_slice(name=SLICE_NAME)
    interfaces = []
    for i in range(NUM_NODES):
        name = f"Node{i + 1}"
        instance_type = MASTER_TYPE if i == 0 else WORKER_TYPE
        node = cluster.add_node(
            name=name, site=SITE, instance_type=instance_type, image=IMAGE
        )
        nic = node.add_component(model="NIC_Basic", name=name)
        interfaces.append(nic.get_interfaces()[0])
    cluster.add_l2network(name=NETWORK_NAME, interfaces=interfaces)
    cluster.submit(progress=False)
    return cluster
# ===========================================================================
#  Step 2 — Assign IPs and bring interfaces up
# ===========================================================================
def configure_interfaces(cluster):
    """Assign static IPs from the subnet and install net-tools on every node."""
    available_ips = list(SUBNET)[1:]
    for node in _ordered_nodes(cluster):
        iface = node.get_interface(network_name=NETWORK_NAME)
        iface.ip_addr_add(addr=available_ips.pop(0), subnet=SUBNET)
        node.execute("sudo apt-get update -qq", quiet=True)
        node.execute("sudo apt-get install -y -qq net-tools", quiet=True)
        node.execute(f"sudo ifconfig {iface.get_os_interface()} up", quiet=True)
# ===========================================================================
#  Step 3 — Build inventory files (hosts, ips, workers, gpu_ips)
# ===========================================================================
def build_inventory(cluster):
    """Collect host metadata and write inventory files to WORK_DIR."""
    host_lines = ["127.0.0.1 localhost"]
    ip_lines, worker_lines, gpu_lines = [], [], []

    for index, node in enumerate(_ordered_nodes(cluster)):
        data_ip = node.execute("hostname -I", quiet=True)[0].split()[1]
        hostname = node.execute("hostname", quiet=True)[0].strip()
        alias = f"node{index + 1}"
        vm_name = f"vm{index}"

        ip_lines.append(data_ip)
        host_lines.append(f"{data_ip} {alias} {vm_name} {hostname}")

        if index > 0:
            worker_lines.append(vm_name)

        gpu_stdout = node.execute(f"lspci | grep {GPU_VENDOR}", quiet=True)[0]
        if GPU_VENDOR in gpu_stdout:
            gpu_lines.append(data_ip)

    HOSTS_FILE.write_text("\n".join(host_lines))
    IPS_FILE.write_text("\n".join(ip_lines))
    WORKERS_FILE.write_text("\n".join(worker_lines))
    if gpu_lines:
        GPU_IPS_FILE.write_text("\n".join(gpu_lines))
# ===========================================================================
#  Step 4 — Generate SSH keypair
# ===========================================================================
def setup_ssh_key():
    """Generate a cluster-wide SSH keypair on the orchestrator node."""
    subprocess.run(
        ["ssh-keygen", "-q", "-t", "rsa", "-N", "", "-f", str(SSH_KEY)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    )
# ===========================================================================
#  Step 5 — Distribute inventory, /etc/hosts, SSH keys, and NAT64 fix
# ===========================================================================
def provision_nodes(cluster):
    """Upload all configuration artifacts to every node in one pass."""
    has_gpu_file = GPU_IPS_FILE.exists()
    for node in _ordered_nodes(cluster):
        # --- /etc/hosts (back up once, then overwrite) ---
        node.execute(
            "[ -f /etc/hosts_backup ] || sudo cp /etc/hosts /etc/hosts_backup",
            quiet=True,
        )
        node.upload_file(str(HOSTS_FILE), "/home/ubuntu/hosts")
        node.upload_file(str(IPS_FILE), "/home/ubuntu/ips.txt")
        node.upload_file(str(WORKERS_FILE), "/home/ubuntu/workers")
        if has_gpu_file:
            node.upload_file(str(GPU_IPS_FILE), "/home/ubuntu/gpu_ips.txt")
        node.execute("sudo cp /home/ubuntu/hosts /etc/hosts", quiet=True)

        # --- SSH keys ---
        node.upload_file(str(SSH_KEY), "/home/ubuntu/.ssh/id_rsa")
        node.upload_file(f"{SSH_KEY}.pub", "/home/ubuntu/.ssh/id_rsa.pub")
        node.execute(
            "cat /home/ubuntu/.ssh/id_rsa.pub >> /home/ubuntu/.ssh/authorized_keys "
            "&& chmod 600 /home/ubuntu/.ssh/id_rsa*",
            quiet=True,
        )
        # --- NAT64 for IPv6-managed nodes ---
        if isinstance(ip_address(node.get_management_ip()), IPv6Address):
            node.upload_file(str(NAT64_SCRIPT), "/home/ubuntu/nat64.sh")
            node.execute(
                "chmod +x /home/ubuntu/nat64.sh && sudo bash /home/ubuntu/nat64.sh",
            )
# ===========================================================================
#  Utility — Renew slice lease
# ===========================================================================
def renew_slice(slice_name, days):
    new_end = (datetime.now(timezone.utc) + timedelta(days=days)).strftime(
        "%Y-%m-%d %H:%M:%S %z"
    )
    fablib.get_slice(name=slice_name).renew(new_end)
# ===========================================================================
#  Orchestrator
# ===========================================================================
def main():
    """Run the full provisioning pipeline and return the ready cluster."""
    cluster = create_slice()
    configure_interfaces(cluster)
    build_inventory(cluster)
    setup_ssh_key()
    provision_nodes(cluster)
    renew_slice(SLICE_NAME,15)
    return cluster
if __name__ == "__main__":
    main()

#### Cluster (C2) - 16 cores, 64 GB RAM, 1000 GB Storage + 2 GPUs (1T4 + 2RTX)

In [ ]:
"""FABRIC cluster provisioning: create, configure, and provision a master/worker cluster.

Provisions a single-site cluster on the FABRIC testbed with one master node and
three worker nodes connected via an L2 network. Handles GPU attachment, IP
assignment, inventory generation, /etc/hosts distribution, SSH key distribution,
and IPv6 NAT64 setup.

Requirements:
    - FABRIC JupyterHub environment (or fabrictestbed-extensions installed with
      valid credentials at ~/.fabric/config).
    - nat64.sh present at WORK_DIR if any node has an IPv6 management address.

Usage (notebook cell):
    cluster = main()                  # provision everything
    renew_slice("C2-TACC", days=15)   # extend lease separately
"""

import subprocess
from datetime import datetime, timedelta, timezone
from ipaddress import IPv4Network, IPv6Address, ip_address
from pathlib import Path

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# ===========================================================================
#  Configuration — edit these for each experiment
# ===========================================================================
NUM_NODES = 4
SLICE_NAME = "C2-TACC"
SITE = "TACC"
MASTER_TYPE = "fabric.c8.m16.d1000"
WORKER_TYPE = "fabric.c16.m64.d100"
IMAGE = "default_ubuntu_20"
NETWORK_NAME = "cluster_network"
SUBNET = IPv4Network("192.168.1.0/24")
GPU_VENDOR = "NVIDIA"

# GPU allocation map: {node_index: [gpu_model, ...]}
#   node 0 = master (no GPU), nodes 1-3 = workers
#   Set to {} for CPU-only clusters (C1, C3, C4).
GPU_ALLOCATION = {
    1: ["GPU_TeslaT4"],                        # Node2 → 1× Tesla T4
    2: ["GPU_RTX6000", "GPU_RTX6000"],         # Node3 → 2× RTX 6000
}

WORK_DIR = Path("/home/fabric/work")
HOSTS_FILE = WORK_DIR / "hosts"
IPS_FILE = WORK_DIR / "ips.txt"
WORKERS_FILE = WORK_DIR / "workers"
GPU_IPS_FILE = WORK_DIR / "gpu_ips.txt"
SSH_KEY = WORK_DIR / "id_rsa"
NAT64_SCRIPT = WORK_DIR / "nat64.sh"

# ===========================================================================
#  Initialise fablib
# ===========================================================================
fablib = fablib_manager()


# ===========================================================================
#  Helper
# ===========================================================================
def _ordered_nodes(cluster):
    """Return nodes sorted by name so Node1 (master) is always index 0."""
    return sorted(cluster.get_nodes(), key=lambda n: n.get_name())


# ===========================================================================
#  Step 1 — Create the slice (with optional GPUs)
# ===========================================================================
def create_slice():
    """Build a master + worker slice, attach GPUs per GPU_ALLOCATION, submit."""
    cluster = fablib.new_slice(name=SLICE_NAME)
    interfaces = []

    for i in range(NUM_NODES):
        name = f"Node{i + 1}"
        instance_type = MASTER_TYPE if i == 0 else WORKER_TYPE
        node = cluster.add_node(
            name=name, site=SITE, instance_type=instance_type, image=IMAGE
        )

        # NIC
        nic = node.add_component(model="NIC_Basic", name=name)
        interfaces.append(nic.get_interfaces()[0])

        # GPUs (skip master; workers only if listed in the allocation map)
        for gpu_idx, gpu_model in enumerate(GPU_ALLOCATION.get(i, [])):
            node.add_component(model=gpu_model, name=f"gpu_{i}_{gpu_idx}")

    cluster.add_l2network(name=NETWORK_NAME, interfaces=interfaces)
    cluster.submit(progress=False)
    return cluster


# ===========================================================================
#  Step 2 — Assign IPs and bring interfaces up
# ===========================================================================
def configure_interfaces(cluster):
    """Assign static IPs from the subnet and install net-tools on every node."""
    available_ips = list(SUBNET)[1:]
    for node in _ordered_nodes(cluster):
        iface = node.get_interface(network_name=NETWORK_NAME)
        iface.ip_addr_add(addr=available_ips.pop(0), subnet=SUBNET)
        node.execute("sudo apt-get update -qq", quiet=True)
        node.execute("sudo apt-get install -y -qq net-tools", quiet=True)
        node.execute(f"sudo ifconfig {iface.get_os_interface()} up", quiet=True)


# ===========================================================================
#  Step 3 — Build inventory files (hosts, ips, workers, gpu_ips)
# ===========================================================================
def build_inventory(cluster):
    """Collect host metadata and write inventory files to WORK_DIR."""
    host_lines = ["127.0.0.1 localhost"]
    ip_lines, worker_lines, gpu_lines = [], [], []

    for index, node in enumerate(_ordered_nodes(cluster)):
        data_ip = node.execute("hostname -I", quiet=True)[0].split()[1]
        hostname = node.execute("hostname", quiet=True)[0].strip()
        alias = f"node{index + 1}"
        vm_name = f"vm{index}"

        ip_lines.append(data_ip)
        host_lines.append(f"{data_ip} {alias} {vm_name} {hostname}")

        if index > 0:
            worker_lines.append(vm_name)

        gpu_stdout = node.execute(f"lspci | grep {GPU_VENDOR}", quiet=True)[0]
        if GPU_VENDOR in gpu_stdout:
            gpu_lines.append(data_ip)

    HOSTS_FILE.write_text("\n".join(host_lines))
    IPS_FILE.write_text("\n".join(ip_lines))
    WORKERS_FILE.write_text("\n".join(worker_lines))
    if gpu_lines:
        GPU_IPS_FILE.write_text("\n".join(gpu_lines))


# ===========================================================================
#  Step 4 — Generate SSH keypair
# ===========================================================================
def setup_ssh_key():
    """Generate a cluster-wide SSH keypair on the orchestrator node."""
    subprocess.run(
        ["ssh-keygen", "-q", "-t", "rsa", "-N", "", "-f", str(SSH_KEY)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    )


# ===========================================================================
#  Step 5 — Distribute inventory, /etc/hosts, SSH keys, and NAT64 fix
# ===========================================================================
def provision_nodes(cluster):
    """Upload all configuration artifacts to every node in one pass."""
    has_gpu_file = GPU_IPS_FILE.exists()

    for node in _ordered_nodes(cluster):
        # --- /etc/hosts (back up once, then overwrite) ---
        node.execute(
            "[ -f /etc/hosts_backup ] || sudo cp /etc/hosts /etc/hosts_backup",
            quiet=True,
        )
        node.upload_file(str(HOSTS_FILE), "/home/ubuntu/hosts")
        node.upload_file(str(IPS_FILE), "/home/ubuntu/ips.txt")
        node.upload_file(str(WORKERS_FILE), "/home/ubuntu/workers")
        if has_gpu_file:
            node.upload_file(str(GPU_IPS_FILE), "/home/ubuntu/gpu_ips.txt")
        node.execute("sudo cp /home/ubuntu/hosts /etc/hosts", quiet=True)

        # --- SSH keys ---
        node.upload_file(str(SSH_KEY), "/home/ubuntu/.ssh/id_rsa")
        node.upload_file(f"{SSH_KEY}.pub", "/home/ubuntu/.ssh/id_rsa.pub")
        node.execute(
            "cat /home/ubuntu/.ssh/id_rsa.pub >> /home/ubuntu/.ssh/authorized_keys "
            "&& chmod 600 /home/ubuntu/.ssh/id_rsa*",
            quiet=True,
        )

        # --- NAT64 for IPv6-managed nodes ---
        if isinstance(ip_address(node.get_management_ip()), IPv6Address):
            node.upload_file(str(NAT64_SCRIPT), "/home/ubuntu/nat64.sh")
            node.execute(
                "chmod +x /home/ubuntu/nat64.sh && sudo bash /home/ubuntu/nat64.sh",
                quiet=True,
            )


# ===========================================================================
#  Utility — Renew slice lease
# ===========================================================================
def renew_slice(slice_name, days=7):
    """Extend an existing slice lease by *days* (FABRIC allows up to 14)."""
    new_end = (datetime.now(timezone.utc) + timedelta(days=days)).strftime(
        "%Y-%m-%d %H:%M:%S %z"
    )
    fablib.get_slice(name=slice_name).renew(new_end)


# ===========================================================================
#  Orchestrator
# ===========================================================================
def main():
    """Run the full provisioning pipeline and return the ready cluster."""
    cluster = create_slice()
    configure_interfaces(cluster)
    build_inventory(cluster)
    setup_ssh_key()
    provision_nodes(cluster)
    renew_slice(SLICE_NAME, days=15)
    return cluster

if __name__ == "__main__":
    main()

#### Cluster (C3) - 16 cores, 64 GB RAM, 1000 GB Storage

In [ ]:
"""FABRIC cluster provisioning: create, configure, and provision a master/worker cluster.
Provisions a single-site cluster on the FABRIC testbed with one master node and three worker nodes connected via an L2 network. Handles IP assignment, inventory generation, 
/etc/hosts distribution, SSH key distribution, and IPv6 NAT64 setup.
Requirements:
    - FABRIC JupyterHub environment (or fabrictestbed-extensions installed with
      valid credentials at ~/.fabric/config).
    - nat64.sh present at WORK_DIR if any node has an IPv6 management address.
Usage (notebook cell):
    cluster = main()              # provision everything
    renew_slice("C1-HAWI", days=15)  # extend lease separately
"""
import subprocess
from datetime import datetime, timedelta, timezone
from ipaddress import IPv4Network, IPv6Address, ip_address
from pathlib import Path
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
# ===========================================================================
#  Configuration — edit these for each experiment
# ===========================================================================
NUM_NODES = 4
SLICE_NAME = "C3-HAWI"
SITE = "HAWI"
MASTER_TYPE = "fabric.c8.m16.d1000"
WORKER_TYPE = "fabric.c16.m64.d100"
IMAGE = "default_ubuntu_20"
NETWORK_NAME = "cluster_network"
SUBNET = IPv4Network("192.168.1.0/24")
GPU_VENDOR = "NVIDIA"

WORK_DIR = Path("/home/fabric/work")
HOSTS_FILE = WORK_DIR / "hosts"
IPS_FILE = WORK_DIR / "ips.txt"
WORKERS_FILE = WORK_DIR / "workers"
GPU_IPS_FILE = WORK_DIR / "gpu_ips.txt"
SSH_KEY = WORK_DIR / "id_rsa"
NAT64_SCRIPT = WORK_DIR / "nat64.sh"
# ===========================================================================
#  Initialise fablib
# ===========================================================================
fablib = fablib_manager()
# ===========================================================================
#  Helper
# ===========================================================================
def _ordered_nodes(cluster):
    """Return nodes sorted by name so Node1 (master) is always index 0."""
    return sorted(cluster.get_nodes(), key=lambda n: n.get_name())
# ===========================================================================
#  Step 1 — Create the slice
# ===========================================================================
def create_slice():
    """Build a master + worker slice on a single FABRIC site and submit."""
    cluster = fablib.new_slice(name=SLICE_NAME)
    interfaces = []
    for i in range(NUM_NODES):
        name = f"Node{i + 1}"
        instance_type = MASTER_TYPE if i == 0 else WORKER_TYPE
        node = cluster.add_node(
            name=name, site=SITE, instance_type=instance_type, image=IMAGE
        )
        nic = node.add_component(model="NIC_Basic", name=name)
        interfaces.append(nic.get_interfaces()[0])
    cluster.add_l2network(name=NETWORK_NAME, interfaces=interfaces)
    cluster.submit(progress=False)
    return cluster
# ===========================================================================
#  Step 2 — Assign IPs and bring interfaces up
# ===========================================================================
def configure_interfaces(cluster):
    """Assign static IPs from the subnet and install net-tools on every node."""
    available_ips = list(SUBNET)[1:]
    for node in _ordered_nodes(cluster):
        iface = node.get_interface(network_name=NETWORK_NAME)
        iface.ip_addr_add(addr=available_ips.pop(0), subnet=SUBNET)
        node.execute("sudo apt-get update -qq", quiet=True)
        node.execute("sudo apt-get install -y -qq net-tools", quiet=True)
        node.execute(f"sudo ifconfig {iface.get_os_interface()} up", quiet=True)
# ===========================================================================
#  Step 3 — Build inventory files (hosts, ips, workers, gpu_ips)
# ===========================================================================
def build_inventory(cluster):
    """Collect host metadata and write inventory files to WORK_DIR."""
    host_lines = ["127.0.0.1 localhost"]
    ip_lines, worker_lines, gpu_lines = [], [], []

    for index, node in enumerate(_ordered_nodes(cluster)):
        data_ip = node.execute("hostname -I", quiet=True)[0].split()[1]
        hostname = node.execute("hostname", quiet=True)[0].strip()
        alias = f"node{index + 1}"
        vm_name = f"vm{index}"

        ip_lines.append(data_ip)
        host_lines.append(f"{data_ip} {alias} {vm_name} {hostname}")

        if index > 0:
            worker_lines.append(vm_name)

        gpu_stdout = node.execute(f"lspci | grep {GPU_VENDOR}", quiet=True)[0]
        if GPU_VENDOR in gpu_stdout:
            gpu_lines.append(data_ip)

    HOSTS_FILE.write_text("\n".join(host_lines))
    IPS_FILE.write_text("\n".join(ip_lines))
    WORKERS_FILE.write_text("\n".join(worker_lines))
    if gpu_lines:
        GPU_IPS_FILE.write_text("\n".join(gpu_lines))
# ===========================================================================
#  Step 4 — Generate SSH keypair
# ===========================================================================
def setup_ssh_key():
    """Generate a cluster-wide SSH keypair on the orchestrator node."""
    subprocess.run(
        ["ssh-keygen", "-q", "-t", "rsa", "-N", "", "-f", str(SSH_KEY)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    )
# ===========================================================================
#  Step 5 — Distribute inventory, /etc/hosts, SSH keys, and NAT64 fix
# ===========================================================================
def provision_nodes(cluster):
    """Upload all configuration artifacts to every node in one pass."""
    has_gpu_file = GPU_IPS_FILE.exists()
    for node in _ordered_nodes(cluster):
        # --- /etc/hosts (back up once, then overwrite) ---
        node.execute(
            "[ -f /etc/hosts_backup ] || sudo cp /etc/hosts /etc/hosts_backup",
            quiet=True,
        )
        node.upload_file(str(HOSTS_FILE), "/home/ubuntu/hosts")
        node.upload_file(str(IPS_FILE), "/home/ubuntu/ips.txt")
        node.upload_file(str(WORKERS_FILE), "/home/ubuntu/workers")
        if has_gpu_file:
            node.upload_file(str(GPU_IPS_FILE), "/home/ubuntu/gpu_ips.txt")
        node.execute("sudo cp /home/ubuntu/hosts /etc/hosts", quiet=True)

        # --- SSH keys ---
        node.upload_file(str(SSH_KEY), "/home/ubuntu/.ssh/id_rsa")
        node.upload_file(f"{SSH_KEY}.pub", "/home/ubuntu/.ssh/id_rsa.pub")
        node.execute(
            "cat /home/ubuntu/.ssh/id_rsa.pub >> /home/ubuntu/.ssh/authorized_keys "
            "&& chmod 600 /home/ubuntu/.ssh/id_rsa*",
            quiet=True,
        )
        # --- NAT64 for IPv6-managed nodes ---
        if isinstance(ip_address(node.get_management_ip()), IPv6Address):
            node.upload_file(str(NAT64_SCRIPT), "/home/ubuntu/nat64.sh")
            node.execute(
                "chmod +x /home/ubuntu/nat64.sh && sudo bash /home/ubuntu/nat64.sh",
                quiet=True,
            )
# ===========================================================================
#  Utility — Renew slice lease
# ===========================================================================
def renew_slice(slice_name, days):
    new_end = (datetime.now(timezone.utc) + timedelta(days=days)).strftime(
        "%Y-%m-%d %H:%M:%S %z"
    )
    fablib.get_slice(name=slice_name).renew(new_end)
# ===========================================================================
#  Orchestrator
# ===========================================================================
def main():
    """Run the full provisioning pipeline and return the ready cluster."""
    cluster = create_slice()
    configure_interfaces(cluster)
    build_inventory(cluster)
    setup_ssh_key()
    provision_nodes(cluster)
    renew_slice(SLICE_NAME,15)
    return cluster
if __name__ == "__main__":
    main()

#### Cluster (C4) - 16 cores, 128 GB RAM, 1000 GB Storage

In [ ]:
"""FABRIC cluster provisioning: create, configure, and provision a master/worker cluster.
Provisions a single-site cluster on the FABRIC testbed with one master node and three worker nodes connected via an L2 network. Handles IP assignment, inventory generation, 
/etc/hosts distribution, SSH key distribution, and IPv6 NAT64 setup.
Requirements:
    - FABRIC JupyterHub environment (or fabrictestbed-extensions installed with
      valid credentials at ~/.fabric/config).
    - nat64.sh present at WORK_DIR if any node has an IPv6 management address.
Usage (notebook cell):
    cluster = main()              # provision everything
    renew_slice("C1-HAWI", days=15)  # extend lease separately
"""
import subprocess
from datetime import datetime, timedelta, timezone
from ipaddress import IPv4Network, IPv6Address, ip_address
from pathlib import Path
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
# ===========================================================================
#  Configuration — edit these for each experiment
# ===========================================================================
NUM_NODES = 4
SLICE_NAME = "C4-MICH"
SITE = "MICH"
MASTER_TYPE = "fabric.c8.m16.d1000"
WORKER_TYPE = "fabric.c16.m128.d100"
IMAGE = "default_ubuntu_20"
NETWORK_NAME = "cluster_network"
SUBNET = IPv4Network("192.168.1.0/24")
GPU_VENDOR = "NVIDIA"
WORK_DIR = Path("/home/fabric/work")
HOSTS_FILE = WORK_DIR / "hosts"
IPS_FILE = WORK_DIR / "ips.txt"
WORKERS_FILE = WORK_DIR / "workers"
GPU_IPS_FILE = WORK_DIR / "gpu_ips.txt"
SSH_KEY = WORK_DIR / "id_rsa"
NAT64_SCRIPT = WORK_DIR / "nat64.sh"
# ===========================================================================
#  Initialise fablib
# ===========================================================================
fablib = fablib_manager()
# ===========================================================================
#  Helper functions
# ===========================================================================
def _ordered_nodes(cluster):
    """Return nodes sorted by name so Node1 (master) is always index 0."""
    return sorted(cluster.get_nodes(), key=lambda n: n.get_name())
# ===========================================================================
#  Step 1 — Create the slice
# ===========================================================================
def create_slice():
    """Build a master + worker slice on a single FABRIC site and submit."""
    cluster = fablib.new_slice(name=SLICE_NAME)
    interfaces = []
    for i in range(NUM_NODES):
        name = f"Node{i + 1}"
        instance_type = MASTER_TYPE if i == 0 else WORKER_TYPE
        node = cluster.add_node(
            name=name, site=SITE, instance_type=instance_type, image=IMAGE
        )
        nic = node.add_component(model="NIC_Basic", name=name)
        interfaces.append(nic.get_interfaces()[0])
    cluster.add_l2network(name=NETWORK_NAME, interfaces=interfaces)
    cluster.submit(progress=False)
    return cluster
# ===========================================================================
#  Step 2 — Assign IPs and bring interfaces up
# ===========================================================================
def configure_interfaces(cluster):
    """Assign static IPs from the subnet and install net-tools on every node."""
    available_ips = list(SUBNET)[1:]
    for node in _ordered_nodes(cluster):
        iface = node.get_interface(network_name=NETWORK_NAME)
        iface.ip_addr_add(addr=available_ips.pop(0), subnet=SUBNET)
        node.execute("sudo apt-get update -qq", quiet=True)
        node.execute("sudo apt-get install -y -qq net-tools", quiet=True)
        node.execute(f"sudo ifconfig {iface.get_os_interface()} up", quiet=True)
# ===========================================================================
#  Step 3 — Build inventory files (hosts, ips, workers, gpu_ips)
# ===========================================================================
def build_inventory(cluster):
    """Collect host metadata and write inventory files to WORK_DIR."""
    host_lines = ["127.0.0.1 localhost"]
    ip_lines, worker_lines, gpu_lines = [], [], []

    for index, node in enumerate(_ordered_nodes(cluster)):
        data_ip = node.execute("hostname -I", quiet=True)[0].split()[1]
        hostname = node.execute("hostname", quiet=True)[0].strip()
        alias = f"node{index + 1}"
        vm_name = f"vm{index}"

        ip_lines.append(data_ip)
        host_lines.append(f"{data_ip} {alias} {vm_name} {hostname}")

        if index > 0:
            worker_lines.append(vm_name)

        gpu_stdout = node.execute(f"lspci | grep {GPU_VENDOR}", quiet=True)[0]
        if GPU_VENDOR in gpu_stdout:
            gpu_lines.append(data_ip)

    HOSTS_FILE.write_text("\n".join(host_lines))
    IPS_FILE.write_text("\n".join(ip_lines))
    WORKERS_FILE.write_text("\n".join(worker_lines))
    if gpu_lines:
        GPU_IPS_FILE.write_text("\n".join(gpu_lines))
# ===========================================================================
#  Step 4 — Generate SSH keypair
# ===========================================================================
def setup_ssh_key():
    """Generate a cluster-wide SSH keypair on the orchestrator node."""
    subprocess.run(
        ["ssh-keygen", "-q", "-t", "rsa", "-N", "", "-f", str(SSH_KEY)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    )
# ===========================================================================
#  Step 5 — Distribute inventory, /etc/hosts, SSH keys, and NAT64 fix
# ===========================================================================
def provision_nodes(cluster):
    """Upload all configuration artifacts to every node in one pass."""
    has_gpu_file = GPU_IPS_FILE.exists()
    for node in _ordered_nodes(cluster):
        # --- /etc/hosts (back up once, then overwrite) ---
        node.execute(
            "[ -f /etc/hosts_backup ] || sudo cp /etc/hosts /etc/hosts_backup",
            quiet=True,
        )
        node.upload_file(str(HOSTS_FILE), "/home/ubuntu/hosts")
        node.upload_file(str(IPS_FILE), "/home/ubuntu/ips.txt")
        node.upload_file(str(WORKERS_FILE), "/home/ubuntu/workers")
        if has_gpu_file:
            node.upload_file(str(GPU_IPS_FILE), "/home/ubuntu/gpu_ips.txt")
        node.execute("sudo cp /home/ubuntu/hosts /etc/hosts", quiet=True)

        # --- SSH keys ---
        node.upload_file(str(SSH_KEY), "/home/ubuntu/.ssh/id_rsa")
        node.upload_file(f"{SSH_KEY}.pub", "/home/ubuntu/.ssh/id_rsa.pub")
        node.execute(
            "cat /home/ubuntu/.ssh/id_rsa.pub >> /home/ubuntu/.ssh/authorized_keys "
            "&& chmod 600 /home/ubuntu/.ssh/id_rsa*",
            quiet=True,
        )
        # --- NAT64 for IPv6-managed nodes ---
        if isinstance(ip_address(node.get_management_ip()), IPv6Address):
            node.upload_file(str(NAT64_SCRIPT), "/home/ubuntu/nat64.sh")
            node.execute(
                "chmod +x /home/ubuntu/nat64.sh && sudo bash /home/ubuntu/nat64.sh"
            )
# ===========================================================================
#  Utility — Renew slice lease
# ===========================================================================
def renew_slice(slice_name, days):
    new_end = (datetime.now(timezone.utc) + timedelta(days=days)).strftime(
        "%Y-%m-%d %H:%M:%S %z"
    )
    fablib.get_slice(name=slice_name).renew(new_end)
# ===========================================================================
#  Orchestrator
# ===========================================================================
def main():
    """Run the full provisioning pipeline and return the ready cluster."""
    cluster = create_slice()
    configure_interfaces(cluster)
    build_inventory(cluster)
    setup_ssh_key()
    provision_nodes(cluster)
    renew_slice(SLICE_NAME,15)
    return cluster
if __name__ == "__main__":
    main()